# Phase 1: Dataset Discovery & EDF Metadata/Channel Audit

This notebook executes the Phase 1 dataset discovery and EDF metadata/channel audit pipeline (`imagined_speech.data.edf_audit`) on Google Colab (or locally).

### Workflow Overview
1. Mount Google Drive.
2. Clone or update the `imagined-speech-eeg` repository.
3. Install the `imagined-speech-recognition` package (`pip install -e ".[dev]"`).
4. Verify `/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip` and extract into `/content/data/raw` without redundant re-extraction.
5. Automatically locate `DATA_ROOT` containing `Char/`, `Digit/`, and `Image/` (even if wrapped in an outer folder).
6. Run `run_dataset_audit` and persist outputs to `/content/drive/MyDrive/imagined_speech_outputs/phase1_audit/`.
7. Display the audit summary and previews of `manifest.csv` and `channel_audit.csv`.
8. Verify that `manifest.csv`, `channel_audit.csv`, and `audit_summary.json` exist and are non-empty.

## 1. Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository

In [ ]:
import subprocess

REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f'Cloning {REPO_URL} into {COLAB_REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO_DIR)], check=True)
    elif (COLAB_REPO_DIR / '.git').is_dir():
        print(f'Updating existing repository at {COLAB_REPO_DIR}...')
        subprocess.run(['git', '-C', str(COLAB_REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        raise RuntimeError(
            f'Directory {COLAB_REPO_DIR} exists but is not a valid Git repository.'
        )
    os.chdir(COLAB_REPO_DIR)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'pyproject.toml not found in working directory {REPO_ROOT}. '
        'Ensure the notebook is running from the repository root.'
    )

print(f'Repository root: {REPO_ROOT}')

## 3. Install the Package

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

# Ensure src/ is immediately importable in the active kernel session
src_dir = str((REPO_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import imagined_speech
print(f'Installed imagined_speech version: {imagined_speech.__version__}')

## 4. Verify Dataset ZIP Archive and Extract Safely

In [ ]:
import zipfile

DRIVE_ZIP_PATH = Path(
    os.environ.get(
        'ISR_DRIVE_ZIP',
        '/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip',
    )
)
EXTRACT_ROOT = Path(
    os.environ.get(
        'ISR_EXTRACT_ROOT',
        '/content/data/raw' if IN_COLAB else './data/raw',
    )
)
OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase1_audit'
        if IN_COLAB
        else './outputs/phase1_audit',
    )
)
CONFIG_PATH = REPO_ROOT / 'configs' / 'default_config.yaml'

existing_edfs = (
    [p for p in EXTRACT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() == '.edf']
    if EXTRACT_ROOT.exists()
    else []
)

if existing_edfs:
    print(
        f'Skipping ZIP extraction: found {len(existing_edfs)} existing .edf files '
        f'under {EXTRACT_ROOT}.'
    )
else:
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(
            f'Dataset ZIP archive not found at {DRIVE_ZIP_PATH}. '
            'Verify that Google Drive is mounted at /content/drive and that '
            'MyDrive/datasets/Imagined_speech_EEG_edf.zip exists.'
        )
    if not DRIVE_ZIP_PATH.is_file() or not zipfile.is_zipfile(DRIVE_ZIP_PATH):
        raise ValueError(
            f'Path {DRIVE_ZIP_PATH} exists but is not a valid ZIP archive.'
        )

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {DRIVE_ZIP_PATH} -> {EXTRACT_ROOT} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)

    extracted_edfs = [
        p for p in EXTRACT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() == '.edf'
    ]
    if not extracted_edfs:
        raise RuntimeError(
            f'Archive {DRIVE_ZIP_PATH} was extracted to {EXTRACT_ROOT}, '
            'but no .edf files were found inside.'
        )
    print(f'Extraction complete: {len(extracted_edfs)} .edf files extracted.')

## 5. Locate Dataset Root (`Char/`, `Digit/`, `Image/`)

In [ ]:
from imagined_speech.exceptions import DatasetDiscoveryError

EXPECTED_TASKS = ('Char', 'Digit', 'Image')


def locate_dataset_root(
    search_root: Path,
    expected_tasks: tuple[str, ...] = EXPECTED_TASKS,
) -> tuple[Path, list[str]]:
    """Locate the directory containing Char/, Digit/, and Image/ subdirectories."""
    if not search_root.exists() or not search_root.is_dir():
        raise DatasetDiscoveryError(
            f'Extraction directory does not exist or is not a directory: {search_root}'
        )

    expected_lower = {t.lower() for t in expected_tasks}
    candidate_dirs = [search_root] + sorted(
        (
            d
            for d in search_root.rglob('*')
            if d.is_dir() and not any(part.startswith('.') or part == '__MACOSX' for part in d.parts)
        ),
        key=lambda p: (len(p.parts), p.as_posix().lower()),
    )

    for cand in candidate_dirs:
        child_dirs = {
            child.name.lower(): child.name
            for child in sorted(cand.iterdir(), key=lambda p: p.name.lower())
            if child.is_dir()
        }
        if expected_lower.issubset(child_dirs.keys()):
            discovered_folders = [child_dirs[t.lower()] for t in expected_tasks]
            return cand.resolve(), discovered_folders

    observed_dirs = [
        p.relative_to(search_root).as_posix()
        for p in candidate_dirs[1:15]
    ]
    raise DatasetDiscoveryError(
        f'Could not find a directory under {search_root} containing all required '
        f'category folders {list(expected_tasks)}. Observed subdirectories: {observed_dirs}'
    )


DATA_ROOT, discovered_categories = locate_dataset_root(EXTRACT_ROOT)
print(f'Resolved DATA_ROOT: {DATA_ROOT}')
print(f'Discovered category folders: {discovered_categories}')
for folder in discovered_categories:
    n_edf = sum(
        1
        for p in (DATA_ROOT / folder).rglob('*')
        if p.is_file() and p.suffix.lower() == '.edf'
    )
    print(f'  - {folder}/: {n_edf} EDF file(s)')

## 6. Run Phase 1 Dataset Discovery and EDF Audit

In [ ]:
from imagined_speech.config import load_config
from imagined_speech.data.edf_audit import run_dataset_audit
from imagined_speech.utils import set_global_seed, setup_logging

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Configuration file not found at {CONFIG_PATH}')

config = load_config(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
)
setup_logging(config.log_level)
set_global_seed(config.seed)

print(f'Running Phase 1 audit on DATA_ROOT={config.data_root}')
print(f'Saving persistent outputs to OUTPUT_DIR={config.output_dir}')
audit_result = run_dataset_audit(config, save_outputs=True)

## 7. Display Audit Summary and Previews of `manifest.csv` and `channel_audit.csv`

In [ ]:
import json

print('=== Phase 1 Audit Summary ===')
print(json.dumps(audit_result.summary, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display

manifest_path = OUTPUT_DIR / 'manifest.csv'
channel_audit_path = OUTPUT_DIR / 'channel_audit.csv'

manifest_df = pd.read_csv(manifest_path)
channel_df = pd.read_csv(channel_audit_path)

print(f'manifest.csv preview ({len(manifest_df)} total rows):')
display(manifest_df.head(10))

print(f'channel_audit.csv preview ({len(channel_df)} total rows):')
display(channel_df.head(10))

print('Channel role counts across tasks:')
display(
    channel_df.groupby(['task_category', 'audited_channel_role', 'raw_channel_name'])
    .size()
    .reset_index(name='count')
)

## 8. Verify Persistent Audit Outputs

In [ ]:
REQUIRED_AUDIT_OUTPUTS = (
    'manifest.csv',
    'channel_audit.csv',
    'audit_summary.json',
)

missing_or_empty: list[str] = []
verification_rows: list[dict[str, object]] = []

for filename in REQUIRED_AUDIT_OUTPUTS:
    artifact_path = OUTPUT_DIR / filename
    exists = artifact_path.is_file()
    size_bytes = artifact_path.stat().st_size if exists else 0
    if not exists or size_bytes == 0:
        missing_or_empty.append(f'{artifact_path} (exists={exists}, size_bytes={size_bytes})')
    verification_rows.append(
        {
            'artifact': filename,
            'path': str(artifact_path),
            'exists': exists,
            'size_bytes': size_bytes,
        }
    )

if missing_or_empty:
    raise RuntimeError(
        'Phase 1 audit output verification failed. Missing or empty files: '
        + ', '.join(missing_or_empty)
    )

if manifest_df.empty:
    raise RuntimeError(f'manifest.csv at {manifest_path} contains zero rows.')
if channel_df.empty:
    raise RuntimeError(f'channel_audit.csv at {channel_audit_path} contains zero rows.')

summary_disk = json.loads((OUTPUT_DIR / 'audit_summary.json').read_text(encoding='utf-8'))
if not summary_disk or summary_disk.get('n_discovered_files', 0) <= 0:
    raise RuntimeError('audit_summary.json is empty or reports zero discovered files.')

print(f'All Phase 1 audit outputs verified in {OUTPUT_DIR}:')
display(pd.DataFrame(verification_rows))